# Qwen3-VL-8B on a video

Runs `Qwen/Qwen3-VL-8B-Instruct` over a local video file on **a single GPU** and prints the model's answer.

**Note:** this environment has `transformers==4.52.1`, which predates Qwen3-VL.
Cell 1 upgrades to `transformers>=4.57` + `qwen-vl-utils` — **restart the kernel after it runs**, then skip it on later runs.


In [1]:
# --- Setup (run once, then RESTART KERNEL) ---
# Qwen3-VL needs transformers >= 4.57.0
# %pip install -q -U "transformers>=4.57.0" accelerate "qwen-vl-utils[decord]"


## Config

Set `GPU_ID`, `VIDEO_PATH` and `PROMPT` here. This cell must run **before** torch is
imported — `CUDA_VISIBLE_DEVICES` is only read when CUDA first initializes.

In [2]:
import os
os.environ["HF_HOME"] = "/data/foundation_models_checkpoints"   # must be set before transformers is imported
GPU_ID = 1                        # which physical GPU to use (nvidia-smi index)
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)

VIDEO_PATH = "lemma/master.mp4"   # path to your video
PROMPT     = "Describe what happens in this video, step by step."

MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# Frame sampling. Lower these if you hit OOM; raise for finer temporal detail.
FPS          = 1.0          # frames sampled per second of video
MAX_FRAMES   = 640           # hard cap on number of frames fed to the model
MAX_PIXELS   = 360 * 420    # per-frame budget (in pixels) after resizing
MAX_NEW_TOKENS = 1024

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print(f"{VIDEO_PATH}  ({os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB)  ->  GPU {GPU_ID}")


lemma/master.mp4  (41.2 MB)  ->  GPU 1


## Load the model

~17 GB in bfloat16, which fits comfortably on one 48 GB RTX 6000 Ada.
Everything goes on `cuda:0`, which is the physical GPU selected above.

In [3]:
import torch
from huggingface_hub import snapshot_download
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

assert torch.cuda.device_count() == 1, "expected exactly one visible GPU"

# Resolve the model to a local folder under HF_HOME, then load from that path so
# nothing falls back to the default cache dir inherited from the VS Code server.
MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=os.path.join(os.environ["HF_HOME"], "hub"))

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map={"": 0},          # whole model on the single visible GPU
    # attn_implementation="flash_attention_2",  # enable if flash-attn is installed
)
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print(torch.cuda.get_device_name(0), "|", model.dtype)
print(f"weights: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")


/data2/hongn/miniconda3/envs/qwen3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignored error while writing commit hash to /data/foundation_models_checkpoints/hub/models--Qwen--Qwen3-VL-8B-Instruct/refs/main: [Errno 13] Permission denied: '/data/foundation_models_checkpoints/hub/models--Qwen--Qwen3-VL-8B-Instruct/refs/main'.
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.35it/s]


NVIDIA RTX 6000 Ada Generation | torch.bfloat16
weights: 17.5 GB


## Build the message and preprocess

`qwen_vl_utils.process_vision_info` decodes the video, samples frames at `FPS`
(capped at `MAX_FRAMES`) and resizes them to the pixel budget.

In [4]:
from qwen_vl_utils import process_vision_info

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": VIDEO_PATH,
                "fps": FPS,
                "max_frames": MAX_FRAMES,
                "max_pixels": MAX_PIXELS,
            },
            {"type": "text", "text": PROMPT},
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
images, videos, video_kwargs = process_vision_info(messages, return_video_kwargs=True)

inputs = processor(
    text=[text],
    images=images,
    videos=videos,
    return_tensors="pt",
    padding=True,
    **video_kwargs,
)
inputs = inputs.to(model.device)

print("frames sampled:", videos[0].shape[0] if videos else 0)
print("input tokens  :", inputs["input_ids"].shape[-1])


qwen-vl-utils using decord to read video.
Qwen3VL requires frame timestamps to construct prompts, but the `fps` of the input video could not be inferred. Probably `video_metadata` was missing from inputs and you passed pre-sampled frames. Defaulting to `fps=24`. Please provide `video_metadata` for more accurate results.


frames sampled: 28
input tokens  : 1699


## Generate

In [5]:
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

# strip the prompt tokens off the front of the generation
trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
answer = processor.batch_decode(
    trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(answer)
print(f"\npeak GPU memory: {torch.cuda.max_memory_allocated(0) / 1e9:.1f} GB")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In the video, two men are in a modern, well-lit living room. The man on the left, wearing a black t-shirt and shorts, is initially standing near a TV stand, holding a red cup. He appears to be drinking from it. The man on the right, dressed in a light-colored cardigan and jeans, is holding a green watering can. He walks toward the center of the room, where a large black couch is situated. He then begins to water a potted plant placed on the floor near the couch. As he waters the plant, he turns slightly to his left, continuing to pour water from the watering can. The man in the black t-shirt remains in the background, still holding his red cup and looking toward the other man. The room is decorated with various items, including a lamp, books, and a tripod in the corner, creating a cozy and lived-in atmosphere. The video captures a casual, everyday moment between the two men in their home.

peak GPU memory: 18.1 GB


## Reusable helper

Ask follow-up questions, or point it at a different video, without reloading the model.

In [6]:
def ask_video(video_path, prompt, fps=FPS, max_frames=MAX_FRAMES,
              max_pixels=MAX_PIXELS, max_new_tokens=MAX_NEW_TOKENS):
    """Run Qwen3-VL on `video_path` with `prompt` and return the generated text."""
    msgs = [{
        "role": "user",
        "content": [
            {"type": "video", "video": video_path, "fps": fps,
             "max_frames": max_frames, "max_pixels": max_pixels},
            {"type": "text", "text": prompt},
        ],
    }]

    txt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids, vkw = process_vision_info(msgs, return_video_kwargs=True)
    batch = processor(text=[txt], images=imgs, videos=vids,
                      return_tensors="pt", padding=True, **vkw).to(model.device)

    with torch.inference_mode():
        out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False)

    out = [o[len(i):] for i, o in zip(batch["input_ids"], out)]
    return processor.batch_decode(out, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]


question = '''
Given video of egocentric perspective,

Describe interactions between camera wearer and other people in the video.
'''

print(ask_video("lemma/fpv1.mp4", question))


The camera wearer is holding a watering can and a cup, and walks around the room while watering plants. They interact with another person who is standing near the TV stand, and they both seem to be engaged in a casual conversation. The other person is holding a red cup and appears to be drinking from it while the camera wearer waters the plants. The camera wearer then walks towards a hanging plant and waters it before returning to the TV stand area. The other person continues to stand near the TV stand, occasionally looking at the camera wearer as they move around the room. The camera wearer then walks towards a small table with a plant on it and waters that plant as well. The other person remains standing near the TV stand, occasionally looking at the camera wearer as they move around the room. The camera wearer then walks back towards the TV stand area, still holding the watering can and cup. The other person continues to stand near the TV stand, occasionally looking at the camera we